# Next Word Prediction — Cornell Movie Dialogues

**Dataset:** Cornell Movie Dialogues (~304k lines of natural movie dialogue)  
**Model:** Stacked LSTM with sparse categorical crossentropy  
**Why Cornell:** Short conversational lines = fewer sequences per line = no RAM explosion  

**Setup on Kaggle:**  
Add dataset before running: `+ Add Data` → search `cornell movie dialogs corpus` → Add

## 1. Imports

In [ ]:
import re
import pickle
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Hyperparameters

In [ ]:
VOCAB_SIZE  = 5000
MAX_LEN     = 15    # dialogue lines are short — 15 tokens is enough context
EMBED_DIM   = 128
LSTM_UNITS  = 128
BATCH_SIZE  = 256
EPOCHS      = 50
SUBSET      = 0.20  # use 20% of lines — ~60k lines, safe RAM on Kaggle

## 3. Load Data

In [ ]:
# Cornell Movie Dialogues — each line format:
# L1045 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ They do not!
# We only need the last field (the actual dialogue text)

LINES_PATH = '/kaggle/input/cornell-movie-dialogs-corpus/movie_lines.txt'

lines = []
with open(LINES_PATH, encoding='utf-8', errors='ignore') as f:
    for line in f:
        parts = line.strip().split(' +++$+++ ')
        if len(parts) == 5 and parts[4].strip():
            lines.append(parts[4].strip())

# Shuffle and take subset
np.random.seed(42)
np.random.shuffle(lines)
lines = lines[:int(len(lines) * SUBSET)]

print(f'Total lines in dataset : {len(lines):,}')
print(f'Sample lines:')
for l in lines[:5]:
    print(f'  {l}')

## 4. Clean Text

In [ ]:
def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s']", ' ', text)  # keep letters, apostrophes, spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

lines = [clean(l) for l in lines]
lines = [l for l in lines if len(l.split()) > 2]  # drop very short lines

print(f'Lines after cleaning: {len(lines):,}')
print(f'Sample:')
for l in lines[:5]:
    print(f'  {l}')

## 5. Tokenization

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(lines)

print(f'Full vocab : {len(tokenizer.word_index):,}')
print(f'Using top  : {VOCAB_SIZE:,} words')

## 6. Sequence Generation

Using **sparse labels** (integer per sample) instead of one-hot encoding.  
One-hot with 60k sequences × vocab 5000 = ~1.1 GB RAM → crash.  
Sparse = 0.2 MB → no crash.

In [ ]:
input_sequences = []

for line in lines:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram = token_list[max(0, i - MAX_LEN): i + 1]
        input_sequences.append(n_gram)

print(f'Total sequences: {len(input_sequences):,}')

padded = np.array(pad_sequences(input_sequences, maxlen=MAX_LEN + 1, padding='pre'))

X = padded[:, :-1]  # context
y = padded[:, -1]   # next word index (sparse)

print(f'X shape : {X.shape}  ({X.nbytes / 1e6:.1f} MB)')
print(f'y shape : {y.shape}  ({y.nbytes / 1e6:.2f} MB)')

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42
)
print(f'Train : {X_train.shape}')
print(f'Val   : {X_val.shape}')

## 7. Model

In [ ]:
model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM),

    # First LSTM — passes full sequence to second LSTM
    LSTM(LSTM_UNITS, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),

    # Second LSTM — compresses sequence into single vector
    LSTM(LSTM_UNITS // 2, dropout=0.2, recurrent_dropout=0.2),

    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),

    Dense(VOCAB_SIZE, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=['accuracy']
)

model.summary()

## 8. Training

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 10. Prediction

In [ ]:
def predict_next_words(model, tokenizer, text, num_words=1, temperature=0.8):
    result = []
    current = text.strip().lower()
    for _ in range(num_words):
        token_list = tokenizer.texts_to_sequences([current])[0]
        token_list = token_list[-MAX_LEN:]
        token_list = pad_sequences([token_list], maxlen=MAX_LEN, padding='pre')
        probs = model.predict(token_list, verbose=0)[0]
        # Temperature sampling — avoids always predicting 'the' or 'i'
        probs = np.log(probs + 1e-7) / temperature
        probs = np.exp(probs) / np.sum(np.exp(probs))
        idx = np.random.choice(len(probs), p=probs)
        word = tokenizer.index_word.get(idx, '')
        if not word:
            break
        result.append(word)
        current += ' ' + word
    return result, current


test_inputs = [
    "i don't know",
    "what are you",
    "we have to",
    "i can't believe",
    "you have to",
]

print(f'{"Input":<30} {"Next 3 words (temp=0.8)"}')
print('-' * 60)
for inp in test_inputs:
    words, _ = predict_next_words(model, tokenizer, inp, num_words=3, temperature=0.8)
    print(f'{inp:<30} {" ".join(words)}')

## 11. Save Artifacts

In [ ]:
model.save('/kaggle/working/next_word_lstm.h5')

with open('/kaggle/working/tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)

with open('/kaggle/working/config.pickle', 'wb') as f:
    pickle.dump({'max_len': MAX_LEN, 'vocab_size': VOCAB_SIZE}, f,
                protocol=pickle.HIGHEST_PROTOCOL)

print('Saved to /kaggle/working/ — download from Output tab')